# Почему-то Зепплиен не сохраняет ноутбуки с аутпутами -- поэтому аутпуты в маркдаунах

In [0]:
%spark.pyspark from pyspark.sql import functions as F
users = spark.createDataFrame(
[ ("u1", "Berlin"),
("u2", "Berlin"),
("u3", "Munich"),
("u4", "Hamburg"), ],
["user_id", "city"] )

orders = spark.createDataFrame(
[ ("o1", "u1", "p1", 2, 10.0),
("o2", "u1", "p2", 1, 30.0),
("o3", "u2", "p1", 1, 10.0),
("o4", "u2", "p3", 5, 7.0),
("o5", "u3", "p2", 3, 30.0),
("o6", "u3", "p3", 1, 7.0),
("o7", "u4", "p1", 10, 10.0), ],
["order_id", "user_id", "product_id", "qty", "price"] )

products = spark.createDataFrame(
[ ("p1", "Ring VOLA"),
("p2", "Ring POROG"),
("p3", "Ring TISHINA"), ],
["product_id", "product_name"] )

users.show()
orders.show()

```
+-------+-------+
|user_id|   city|
+-------+-------+
|     u1| Berlin|
|     u2| Berlin|
|     u3| Munich|
|     u4|Hamburg|
+-------+-------+

+--------+-------+----------+---+-----+
|order_id|user_id|product_id|qty|price|
+--------+-------+----------+---+-----+
|      o1|     u1|        p1|  2| 10.0|
|      o2|     u1|        p2|  1| 30.0|
|      o3|     u2|        p1|  1| 10.0|
|      o4|     u2|        p3|  5|  7.0|
|      o5|     u3|        p2|  3| 30.0|
|      o6|     u3|        p3|  1|  7.0|
|      o7|     u4|        p1| 10| 10.0|
+--------+-------+----------+---+-----+


In [1]:
%spark.pyspark
products.show()

```
+----------+------------+
|product_id|product_name|
+----------+------------+
|        p1|   Ring VOLA|
|        p2|  Ring POROG|
|        p3|Ring TISHINA|
+----------+------------+

In [2]:
%spark.pyspark
df_joined = orders.join(users, on="user_id", how="left") \
                  .join(products, on="product_id", how="left") \
                  .withColumn("revenue", F.col("qty") * F.col("price"))

In [3]:
%spark.pyspark
df_joined.show()

```
+----------+-------+--------+---+-----+-------+------------+-------+
|product_id|user_id|order_id|qty|price|   city|product_name|revenue|
+----------+-------+--------+---+-----+-------+------------+-------+
|        p2|     u3|      o5|  3| 30.0| Munich|  Ring POROG|   90.0|
|        p3|     u3|      o6|  1|  7.0| Munich|Ring TISHINA|    7.0|
|        p1|     u4|      o7| 10| 10.0|Hamburg|   Ring VOLA|  100.0|
|        p3|     u2|      o4|  5|  7.0| Berlin|Ring TISHINA|   35.0|
|        p1|     u1|      o1|  2| 10.0| Berlin|   Ring VOLA|   20.0|
|        p2|     u1|      o2|  1| 30.0| Berlin|  Ring POROG|   30.0|
|        p1|     u2|      o3|  1| 10.0| Berlin|   Ring VOLA|   10.0|
+----------+-------+--------+---+-----+-------+------------+-------+

In [4]:
%spark.pyspark
df_agg = df_joined.groupBy("city", "product_id", "product_name") \
    .agg(
        F.count("order_id").alias("orders_cnt"),
        F.sum("qty").alias("qty_sum"),
        F.sum("revenue").alias("revenue_sum")
    )

df_agg.show()

```
+-------+----------+------------+----------+-------+-----------+
|   city|product_id|product_name|orders_cnt|qty_sum|revenue_sum|
+-------+----------+------------+----------+-------+-----------+
| Berlin|        p1|   Ring VOLA|         2|      3|       30.0|
|Hamburg|        p1|   Ring VOLA|         1|     10|      100.0|
| Berlin|        p2|  Ring POROG|         1|      1|       30.0|
| Munich|        p3|Ring TISHINA|         1|      1|        7.0|
| Munich|        p2|  Ring POROG|         1|      3|       90.0|
| Berlin|        p3|Ring TISHINA|         1|      5|       35.0|
+-------+----------+------------+----------+-------+-----------+

In [5]:
%spark.pyspark
from pyspark.sql.window import Window


window_spec = Window.partitionBy("city").orderBy(F.col("revenue_sum").desc())

df_top_products = df_agg.withColumn("rn", F.row_number().over(window_spec)) \
                        .filter(F.col("rn") <= 2) \
                        .drop("rn")


df_top_products.show()

```
+-------+----------+------------+----------+-------+-----------+
|   city|product_id|product_name|orders_cnt|qty_sum|revenue_sum|
+-------+----------+------------+----------+-------+-----------+
| Berlin|        p3|Ring TISHINA|         1|      5|       35.0|
| Berlin|        p1|   Ring VOLA|         2|      3|       30.0|
|Hamburg|        p1|   Ring VOLA|         1|     10|      100.0|
| Munich|        p2|  Ring POROG|         1|      3|       90.0|
| Munich|        p3|Ring TISHINA|         1|      1|        7.0|
+-------+----------+------------+----------+-------+-----------+

In [6]:
%spark.pyspark
output_path = "/tmp/sandbox_zeppelin/mart_city_top_products/"

df_top_products.write \
    .mode("overwrite") \
    .parquet(output_path)


df_read = spark.read.parquet(output_path)

df_read.show()

```
+-------+----------+------------+----------+-------+-----------+
|   city|product_id|product_name|orders_cnt|qty_sum|revenue_sum|
+-------+----------+------------+----------+-------+-----------+
| Berlin|        p3|Ring TISHINA|         1|      5|       35.0|
| Berlin|        p1|   Ring VOLA|         2|      3|       30.0|
|Hamburg|        p1|   Ring VOLA|         1|     10|      100.0|
| Munich|        p2|  Ring POROG|         1|      3|       90.0|
| Munich|        p3|Ring TISHINA|         1|      1|        7.0|
+-------+----------+------------+----------+-------+-----------+